# Parte 4: CNN Híbrida — Feature Vector Completo del Paper

**Este notebook es el método completo de Chao et al. (2022).**

## Diferencia vs Notebook 1

| | Notebook 1 | Notebook 4 (este) |
|---|---|---|
| Features | `[W, X_s, X_v, T]` | `[W, X_s, X̂_s, X̂_v, θ̂]` |
| Dimensión | 42 | 56 |
| θ usado | T verdadero (del HDF5) | θ̂ estimado por UKF |
| X̂_s, X̂_v | X_v real del HDF5 | Predicciones del surrogate con θ̂ |
| Tipo | Upper bound | Método real del paper |

**Por qué 56 y no 42:** el paper usa `[W | X_s | X̂_s | X̂_v | θ̂]`. Incluye tanto los sensores reales `X_s` como los predichos por el surrogate `X̂_s`. La diferencia `X_s - X̂_s` es la **innovación** del UKF — le dice a la CNN cuánto difieren los sensores reales del estado de salud estimado, que es información directamente relacionada con la incertidumbre de la estimación.

## Inputs requeridos
```
Del HDF5:              W, X_s, Y, A  (dev y test)
Del notebook 3 (UKF):  theta_hat_dev.npy   → θ̂(t)
                       theta_hat_test.npy
                       xhat_dev.npy        → [X̂_s(t), X̂_v(t)]
                       xhat_test.npy
```

## Pipeline completo
```
HDF5 → [W, X_s] ─────────────────────────────────────────┐
                                                           ├→ concatenar → CNN → RUL
UKF outputs: [θ̂, X̂_s, X̂_v] ──────────────────────────────┘
```

---
## Celda 1: Imports y dispositivo

In [ ]:
import os
import h5py
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'✓ GPU CUDA: {torch.cuda.get_device_name(0)}')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('✓ Apple MPS')
else:
    DEVICE = torch.device('cpu')
    print('✓ CPU')

print(f'  Dispositivo: {DEVICE} | PyTorch {torch.__version__}')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

---
## Celda 2: Hiperparámetros

In [ ]:
FILENAME  = 'N-CMAPSS_DS02-006.h5'

# ─── Preprocesamiento ─────────────────────────────────────────────────────────
WINDOW_SIZE = 50
STRIDE      = 1
R_EARLY     = 125

# ─── CNN (idéntica al notebook 1, solo cambia n_features) ────────────────────
N_FILTERS   = 10
KERNEL_SIZE = 10
FC_UNITS    = 50

# ─── Entrenamiento ────────────────────────────────────────────────────────────
BATCH_SIZE    = 1024
LEARNING_RATE = 1e-3
MAX_EPOCHS    = 100
PATIENCE      = 5

print('Hiperparámetros:')
print(f'  Ventana: {WINDOW_SIZE} | R_early: {R_EARLY} | Batch: {BATCH_SIZE}')

---
## Celda 3: Cargar datos del HDF5 y outputs del UKF

In [ ]:
def decode_varnames(hdf, key):
    raw = np.array(hdf.get(key))
    if raw.dtype.kind in ('S', 'O'):
        return [v.decode('utf-8') if isinstance(v, bytes) else str(v) for v in raw]
    return list(np.array(raw, dtype='U20'))

# ─── HDF5 ─────────────────────────────────────────────────────────────────────
t0 = time.time()
with h5py.File(FILENAME, 'r') as hdf:
    W_dev    = np.array(hdf.get('W_dev'),    dtype=np.float32)
    X_s_dev  = np.array(hdf.get('X_s_dev'),  dtype=np.float32)
    Y_dev    = np.array(hdf.get('Y_dev'),    dtype=np.float32)
    A_dev    = np.array(hdf.get('A_dev'),    dtype=np.float32)

    W_test   = np.array(hdf.get('W_test'),   dtype=np.float32)
    X_s_test = np.array(hdf.get('X_s_test'), dtype=np.float32)
    Y_test   = np.array(hdf.get('Y_test'),   dtype=np.float32)
    A_test   = np.array(hdf.get('A_test'),   dtype=np.float32)

    W_var   = decode_varnames(hdf, 'W_var')
    X_s_var = decode_varnames(hdf, 'X_s_var')

print(f'HDF5 cargado en {time.time()-t0:.1f}s')

# ─── Outputs del UKF (notebook 3) ────────────────────────────────────────────
theta_hat_dev  = np.load('theta_hat_dev.npy')   # (N_dev,  10)  θ̂ estimado
theta_hat_test = np.load('theta_hat_test.npy')  # (N_test, 10)
xhat_dev       = np.load('xhat_dev.npy')        # (N_dev,  28)  [X̂_s, X̂_v]
xhat_test      = np.load('xhat_test.npy')       # (N_test, 28)

# Separar X̂_s y X̂_v
XS_DIM = X_s_dev.shape[1]   # 14
XV_DIM = xhat_dev.shape[1] - XS_DIM  # 14
T_DIM  = theta_hat_dev.shape[1]      # 10

Xhat_s_dev  = xhat_dev[:, :XS_DIM]    # (N_dev,  14)  X̂_s
Xhat_v_dev  = xhat_dev[:, XS_DIM:]    # (N_dev,  14)  X̂_v
Xhat_s_test = xhat_test[:, :XS_DIM]   # (N_test, 14)
Xhat_v_test = xhat_test[:, XS_DIM:]   # (N_test, 14)

units_dev  = np.unique(A_dev[:, 0]).astype(int)
units_test = np.unique(A_test[:, 0]).astype(int)

print(f'\nOutputs del UKF cargados:')
print(f'  theta_hat_dev  : {theta_hat_dev.shape}')
print(f'  theta_hat_test : {theta_hat_test.shape}')
print(f'  Xhat_s_dev     : {Xhat_s_dev.shape}')
print(f'  Xhat_v_dev     : {Xhat_v_dev.shape}')
print(f'\nUnidades dev : {units_dev}')
print(f'Unidades test: {units_test}')

---
## Celda 4: Construcción del feature vector completo (56 dimensiones)

```
x = [W(4) | X_s(14) | X̂_s(14) | X̂_v(14) | θ̂(10)]  =  56 features
         ↑ sensores reales      ↑ predichos por surrogate con θ̂ del UKF
```

La diferencia `X_s - X̂_s` (innovación) es implícitamente aprendida por la CNN como parte de los 28 valores de sensores que recibe. Incluir ambos le permite a la red aprender no solo el nivel de degradación (θ̂) sino también cuánto se desvía el motor real del modelo físico.

In [ ]:
def build_features(W, X_s, Xhat_s, Xhat_v, theta_hat, Y, R_early=125):
    """
    Construye el feature vector del paper: [W | X_s | X̂_s | X̂_v | θ̂]
    y aplica RUL capping a R_early.
    """
    features = np.concatenate([W, X_s, Xhat_s, Xhat_v, theta_hat], axis=1)
    rul = np.clip(Y.flatten(), 0, R_early)
    return features.astype(np.float32), rul.astype(np.float32)

X_dev_raw,  Y_dev_cap  = build_features(
    W_dev,  X_s_dev,  Xhat_s_dev,  Xhat_v_dev,  theta_hat_dev,  Y_dev,  R_EARLY)
X_test_raw, Y_test_cap = build_features(
    W_test, X_s_test, Xhat_s_test, Xhat_v_test, theta_hat_test, Y_test, R_EARLY)

N_FEATURES = X_dev_raw.shape[1]  # 56

print(f'Feature vector: {N_FEATURES} dimensiones')
print(f'  W       :  4  (alt, Mach, TRA, T2)')
print(f'  X_s     : {XS_DIM}  (sensores físicos reales)')
print(f'  X̂_s     : {XS_DIM}  (sensores predichos por surrogate D con θ̂)')
print(f'  X̂_v     : {XV_DIM}  (sensores virtuales predichos)')
print(f'  θ̂       : {T_DIM}  (parámetros de salud estimados por UKF)')
print(f'  ─────────────────')
print(f'  Total   : {N_FEATURES}')
print(f'\nX_dev_raw  shape: {X_dev_raw.shape}')
print(f'X_test_raw shape: {X_test_raw.shape}')

---
## Celda 5: Normalización

In [ ]:
scaler = MinMaxScaler(feature_range=(-1, 1))
X_dev_norm  = scaler.fit_transform(X_dev_raw)
X_test_norm = scaler.transform(X_test_raw)

print(f'Normalización completada (min-max → [-1, 1])')
print(f'  Rango X_dev_norm  : [{X_dev_norm.min():.2f}, {X_dev_norm.max():.2f}]')
print(f'  Rango X_test_norm : [{X_test_norm.min():.4f}, {X_test_norm.max():.4f}]')

---
## Celda 6: División train / validación

In [ ]:
VAL_UNIT   = units_dev[-1]
mask_val   = A_dev[:, 0] == VAL_UNIT
mask_train = ~mask_val

X_train = X_dev_norm[mask_train]
Y_train = Y_dev_cap[mask_train]
A_train = A_dev[mask_train]

X_val   = X_dev_norm[mask_val]
Y_val   = Y_dev_cap[mask_val]
A_val   = A_dev[mask_val]

train_units = [u for u in units_dev if u != VAL_UNIT]
print(f'Train : unidades {train_units} → {X_train.shape[0]:,} timesteps')
print(f'Val   : unidad   {VAL_UNIT}    → {X_val.shape[0]:,} timesteps')
print(f'Test  : unidades {list(units_test)} → {X_test_norm.shape[0]:,} timesteps')

---
## Celda 7: Dataset con ventana deslizante

In [ ]:
class NCMAPSSDataset(Dataset):
    """
    Genera ventanas ON-THE-FLY — no pre-almacena nada en RAM.
    
    En lugar de guardar 4.4M tensores de (50,56), solo guarda:
      - El array X completo: ~2.3 GB
      - Una lista de índices (unit_id, start): ~35 MB
    
    RAM usada: ~2.4 GB vs 49 GB de la versión anterior.
    Overhead por acceso: mínimo (~1μs de slicing numpy).
    """
    def __init__(self, X, Y, A, window_size=50, stride=1):
        self.X = X   # array completo, no se copia
        self.Y = Y
        self.window_size = window_size
        
        # Solo guardamos los índices de inicio de cada ventana
        self.indices = []
        for unit in np.unique(A[:, 0]):
            mask    = A[:, 0] == unit
            idx_all = np.where(mask)[0]
            n       = len(idx_all)
            for s in range(0, n - window_size + 1, stride):
                self.indices.append((idx_all[s], idx_all[s + window_size - 1]))
        
        ram_indices_mb = len(self.indices) * 2 * 8 / 1e6
        print(f'  Dataset: {len(self.indices):,} ventanas  '
              f'| RAM índices: {ram_indices_mb:.0f} MB  '
              f'| RAM X: {X.nbytes/1e9:.1f} GB')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        start, end = self.indices[i]
        x = torch.tensor(self.X[start:end+1], dtype=torch.float32)
        y = torch.tensor(self.Y[end],          dtype=torch.float32)
        return x, y


print('Construyendo datasets...')
print('Train:'); train_ds = NCMAPSSDataset(X_train, Y_train, A_train, WINDOW_SIZE, STRIDE)
print('Val:');   val_ds   = NCMAPSSDataset(X_val,   Y_val,   A_val,   WINDOW_SIZE, stride=10)
print('Test:');  test_ds  = NCMAPSSDataset(X_test_norm, Y_test_cap, A_test, WINDOW_SIZE, STRIDE)

N_WORKERS = 0 if str(DEVICE) in ['cpu', 'mps'] else 4
PIN_MEM   = (str(DEVICE) == 'cuda')

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=N_WORKERS, pin_memory=PIN_MEM)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=N_WORKERS, pin_memory=PIN_MEM)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=N_WORKERS, pin_memory=PIN_MEM)

xb, yb = next(iter(train_loader))
print(f'\nBatch shape: x={xb.shape}  y={yb.shape}')
print(f'  x debe ser (batch, {WINDOW_SIZE}, {N_FEATURES})')

---
## Celda 8: Arquitectura CNN

Exactamente la misma arquitectura que el notebook 1. Lo único que cambia es `n_features=56` en lugar de 42. La CNN no sabe ni le importa qué features son físicas o estimadas — simplemente aprende a extraer patrones temporales de las 56 dimensiones.

In [ ]:
class RUL_CNN(nn.Module):
    """
    CNN 1D para predicción de RUL — Chao et al. 2022.
    Input:  (batch, window_size, n_features)
    Output: (batch, 1)
    """
    def __init__(self, n_features, window_size, n_filters=10, kernel_size=10, fc_units=50):
        super().__init__()
        pad = (kernel_size - 1) // 2
        self.conv_block = nn.Sequential(
            nn.Conv1d(n_features, n_filters, kernel_size, padding=pad), nn.ReLU(),
            nn.Conv1d(n_filters,  n_filters, kernel_size, padding=pad), nn.ReLU(),
            nn.Conv1d(n_filters,  1,         kernel_size, padding=pad), nn.ReLU(),
        )
        conv_len = self._conv_output_length(window_size, kernel_size, pad)
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv_len, fc_units), nn.ReLU(),
            nn.Linear(fc_units, 1)
        )
        n_params = sum(p.numel() for p in self.parameters())
        print(f'CNN: {n_features}→[{n_filters}×3]→FC({fc_units})→1  |  {n_params:,} params')

    def _conv_output_length(self, length, kernel_size, pad):
        for _ in range(3):
            length = (length + 2*pad - kernel_size) + 1
        return length

    def forward(self, x):
        x = x.permute(0, 2, 1)   # (batch, n_features, window_size)
        x = self.conv_block(x)
        return self.regressor(x)


model = RUL_CNN(
    n_features=N_FEATURES,
    window_size=WINDOW_SIZE,
    n_filters=N_FILTERS,
    kernel_size=KERNEL_SIZE,
    fc_units=FC_UNITS
).to(DEVICE)

# Prueba
with torch.no_grad():
    out = model(torch.randn(4, WINDOW_SIZE, N_FEATURES).to(DEVICE))
    print(f'Forward pass OK: (4,{WINDOW_SIZE},{N_FEATURES}) → {out.shape} ✓')

---
## Celda 9: Entrenamiento

In [ ]:
def train_model(model, train_loader, val_loader, device,
                lr=1e-3, max_epochs=100, patience=5,
                checkpoint_path='best_hybrid_cnn.pt'):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, amsgrad=True)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True)

    history = {'train_loss': [], 'val_loss': []}
    best_val  = float('inf')
    patience_count = 0

    print(f'Entrenando en {device}')
    print(f'{"Época":>6}  {"Train RMSE":>11}  {"Val RMSE":>10}')
    print('-' * 35)

    for epoch in range(1, max_epochs + 1):
        # ── Train ────────────────────────────────────────────────────────────
        model.train()
        t_sum, n_t = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device).unsqueeze(1)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            t_sum += loss.item(); n_t += 1
        train_loss = t_sum / n_t

        # ── Validation ───────────────────────────────────────────────────────
        model.eval()
        v_sum, n_v = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device).unsqueeze(1)
                v_sum += criterion(model(xb), yb).item(); n_v += 1
        val_loss = v_sum / n_v
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            patience_count = 0
            torch.save({'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'val_loss': val_loss}, checkpoint_path)
            marker = ' ◀'
        else:
            patience_count += 1
            marker = f' ({patience_count}/{patience})'

        print(f'{epoch:>6}  {train_loss**0.5:>11.4f}  {val_loss**0.5:>10.4f}{marker}')

        if patience_count >= patience:
            print(f'\nEarly stopping — mejor RMSE val: {best_val**0.5:.4f}')
            break

    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'\nMejor modelo: época {ckpt["epoch"]}')
    return history


history = train_model(
    model, train_loader, val_loader, DEVICE,
    lr=LEARNING_RATE, max_epochs=MAX_EPOCHS, patience=PATIENCE
)

---
## Celda 10: Evaluación en Test Set

In [ ]:
def get_predictions(model, loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.extend(model(xb.to(device)).cpu().numpy().flatten())
            targets.extend(yb.numpy().flatten())
    return np.array(preds), np.array(targets)

def prediction_horizon(rul_true, rul_pred, threshold=5):
    ph = 0
    for v in reversed(np.abs(rul_pred - rul_true) <= threshold):
        if v: ph += 1
        else: break
    return ph


y_pred_test, y_true_test = get_predictions(model, test_loader, DEVICE)

rmse_test = np.sqrt(mean_squared_error(y_true_test, y_pred_test))
mae_test  = np.mean(np.abs(y_pred_test - y_true_test))

print(f'=== Métricas Test Set — CNN Híbrida (paper completo) ===')
print(f'  RMSE : {rmse_test:.4f} ciclos')
print(f'  MAE  : {mae_test:.4f} ciclos')

print(f'\n=== Prediction Horizon por unidad (umbral = 5 ciclos) ===')
window_idx = 0
ph_list, rmse_list = [], []
for unit in units_test:
    mask = A_test[:, 0] == unit
    n_windows = max(0, mask.sum() - WINDOW_SIZE + 1)
    p = y_pred_test[window_idx:window_idx + n_windows]
    t = y_true_test[window_idx:window_idx + n_windows]
    window_idx += n_windows
    if len(p) == 0: continue
    rmse_u = np.sqrt(mean_squared_error(t, p))
    ph_u   = prediction_horizon(t, p)
    rmse_list.append(rmse_u)
    ph_list.append(ph_u)
    print(f'  Unidad {unit:2d}: RMSE={rmse_u:.2f}  |  PH={ph_u}')

print(f'\n  PH promedio: {np.mean(ph_list):.1f}')

---
## Celda 11: Comparación completa — Notebook 1 vs Notebook 4

Aquí cargamos el modelo del notebook 1 (si fue guardado como `rul_cnn_final.pt`) para comparar los dos enfoques directamente.

In [ ]:
# Intentar cargar resultados del notebook 1
nb1_rmse = None
try:
    ckpt_nb1 = torch.load('rul_cnn_final.pt', map_location='cpu')
    nb1_rmse = ckpt_nb1.get('rmse_test', None)
    print(f'Notebook 1 cargado — RMSE test: {nb1_rmse:.4f}' if nb1_rmse else
          'Notebook 1 cargado (sin RMSE guardado)')
except FileNotFoundError:
    print('rul_cnn_final.pt no encontrado — ingresa el RMSE del notebook 1 manualmente:')
    # nb1_rmse = 6.5  # <── descomenta y ajusta con tu resultado real

print()
print('=' * 65)
print('  COMPARACIÓN DE RESULTADOS')
print('=' * 65)
print(f'  {"Modelo":<40}  {"RMSE":>8}  {"PH":>6}')
print('  ' + '-' * 60)
print(f'  {"CNN puro (data-driven, paper)":<40}  {"~7.2":>8}  {"~10":>6}')
if nb1_rmse:
    print(f'  {"CNN + T verdadero (Notebook 1, upper bound)":<40}  {nb1_rmse:>8.2f}  {"?":>6}')
print(f'  {"CNN híbrida + UKF (Notebook 4, paper real)":<40}  {rmse_test:>8.2f}  {np.mean(ph_list):>6.1f}')
print(f'  {"CNN híbrida (paper reportado)":<40}  {"~6.0":>8}  {"~23":>6}')
print('=' * 65)

if nb1_rmse:
    diff = nb1_rmse - rmse_test
    pct  = 100 * diff / nb1_rmse
    print(f'\n  Mejora vs Notebook 1: {diff:+.2f} ciclos ({pct:+.1f}%)')
    if diff > 0:
        print('  ✓ El método completo del paper mejora sobre el upper bound')
        print('    (esto puede pasar si la CNN aprovecha la innovación X_s - X̂_s)')
    else:
        print('  El upper bound (T verdadero) sigue siendo mejor — esperado si el')
        print('  UKF no convergió perfectamente a θ verdadero.')

---
## Celda 12: Visualización de predicciones por motor

In [ ]:
fig, axes = plt.subplots(1, len(units_test), figsize=(6 * len(units_test), 5))
if len(units_test) == 1:
    axes = [axes]

window_idx = 0
for i, unit in enumerate(units_test):
    mask = A_test[:, 0] == unit
    n_windows = max(0, mask.sum() - WINDOW_SIZE + 1)
    p = y_pred_test[window_idx:window_idx + n_windows]
    t = y_true_test[window_idx:window_idx + n_windows]
    window_idx += n_windows

    ts = np.arange(len(t))
    ax = axes[i]
    ax.plot(ts, t, 'k-',  lw=1.5, label='RUL verdadero')
    ax.plot(ts, p, 'b--', lw=1.2, label='RUL predicho', alpha=0.85)
    ax.fill_between(ts, t-5, t+5, alpha=0.12, color='green', label='±5 ciclos')
    rmse_u = np.sqrt(mean_squared_error(t, p))
    ax.set_title(f'Unidad {unit} [TEST]\nRMSE={rmse_u:.2f}')
    ax.set_xlabel('Timestep'); ax.set_ylabel('RUL [ciclos]')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('CNN Híbrida — Feature vector completo del paper [W|X_s|X̂_s|X̂_v|θ̂]',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## Celda 13: Curva de aprendizaje

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, np.sqrt(history['train_loss']), label='Train RMSE', color='steelblue')
ax.plot(epochs, np.sqrt(history['val_loss']),   label='Val RMSE',   color='orange')
ax.axhline(rmse_test, color='red', linestyle=':', alpha=0.8, label=f'Test RMSE={rmse_test:.2f}')
ax.set_xlabel('Época'); ax.set_ylabel('RMSE [ciclos]')
ax.set_title('CNN Híbrida — Curva de aprendizaje')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Celda 14: Guardar modelo final

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'hyperparams': {
        'n_features' : N_FEATURES,
        'window_size': WINDOW_SIZE,
        'n_filters'  : N_FILTERS,
        'kernel_size': KERNEL_SIZE,
        'fc_units'   : FC_UNITS,
    },
    'metrics': {
        'rmse_test': float(rmse_test),
        'mae_test' : float(mae_test),
        'ph_mean'  : float(np.mean(ph_list)),
    },
    'history': history,
}, 'rul_hybrid_cnn_final.pt')

print('Modelo guardado en rul_hybrid_cnn_final.pt')
print(f'  RMSE test : {rmse_test:.4f} ciclos')
print(f'  PH medio  : {np.mean(ph_list):.1f} timesteps')

---
## Resumen del pipeline completo

```
┌─────────────────────────────────────────────────────────────────────┐
│  PIPELINE COMPLETO — Chao et al. (2022)                             │
│                                                                     │
│  N-CMAPSS HDF5                                                      │
│       │                                                             │
│       ├── [W, T] ──→ Surrogate D ──→ [X̂_s, X̂_v]                   │
│       │              (Notebook 2)                                   │
│       │                                                             │
│       ├── [W, X_s] + Surrogate D ──→ UKF ──→ θ̂(t)                  │
│       │                              (Notebook 3)                  │
│       │                                                             │
│       └── [W | X_s | X̂_s | X̂_v | θ̂] ──→ CNN ──→ RUL              │
│                                           (Notebook 4)             │
│                                                                     │
│  Notebook 1 = upper bound (usa T verdadero en lugar de θ̂)          │
└─────────────────────────────────────────────────────────────────────┘
```